# K2Think AIDP GPU Compute Demo

**Custom AI Agent Wrapper optimized for Decentralized Compute**

This notebook demonstrates:
1. 🎮 GPU detection and monitoring
2. 🤖 K2Think AI inference with real GPU logging
3. 📊 AIDP decentralized compute integration

## ⚠️ Important: Enable GPU!

Before running:
1. Click `Runtime` → `Change runtime type`
2. Select **GPU** as Hardware accelerator
3. Click `Save`
4. Come back here and run all cells

---

## Setup: Install Dependencies

In [ ]:
!pip install -q requests python-dotenv
print("✓ Dependencies installed")

## Check GPU Availability

In [ ]:
import subprocess
import sys
import os

# Check if GPU is available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print("🎮 GPU Available:")
    print(result.stdout)
except:
    print("❌ GPU not available - please enable GPU in runtime settings")

## Load GPU Monitor Module

In [ ]:
from datetime import datetime
from typing import Dict
import subprocess

class GPUMonitor:
    """Monitors GPU utilization using nvidia-smi"""
    
    def __init__(self, log_file: str = "gpu-usage.log"):
        self.log_file = log_file
        self.is_available = self._check_gpu_availability()
        self.logs = []
    
    def _check_gpu_availability(self) -> bool:
        """Check if GPU is available"""
        try:
            subprocess.run(['nvidia-smi', '--version'], 
                         capture_output=True, check=True, timeout=5)
            return True
        except:
            return False
    
    def _run_nvidia_smi(self, query: str) -> str:
        """Run nvidia-smi with query"""
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=' + query, '--format=csv,noheader'],
                capture_output=True,
                text=True,
                timeout=10
            )
            return result.stdout.strip()
        except Exception as e:
            return f"Error: {e}"
    
    def log_pre_compute_status(self, task_name: str = "AI Inference"):
        """Log GPU status before compute"""
        timestamp = datetime.now().isoformat()
        separator = "=" * 60
        
        log_entry = f"\n{separator}\n[{timestamp}] PRE-COMPUTE GPU STATUS: {task_name}\n{separator}\n"
        
        if self.is_available:
            gpu_details = self._run_nvidia_smi("index,name,driver_version,memory.total")
            log_entry += f"GPU Details:\n{gpu_details}\n"
            
            metrics = self._run_nvidia_smi("index,memory.used,memory.free,utilization.gpu,temperature.gpu")
            log_entry += f"\nMemory & Utilization:\n{metrics}\n"
        else:
            log_entry += "GPU not available\n"
        
        self._append_log(log_entry)
        print(log_entry)
    
    def log_compute_status(self, task_name: str = "Processing"):
        """Log GPU status during compute"""
        timestamp = datetime.now().isoformat()
        log_entry = f"[{timestamp}] DURING: {task_name}\n"
        
        if self.is_available:
            metrics = self._run_nvidia_smi("index,utilization.gpu,utilization.memory,memory.used,temperature.gpu")
            log_entry += metrics + "\n"
        
        self._append_log(log_entry)
        print(log_entry)
    
    def log_post_compute_status(self, summary: str = ""):
        """Log GPU status after compute"""
        timestamp = datetime.now().isoformat()
        separator = "=" * 60
        
        log_entry = f"\n{separator}\n[{timestamp}] POST-COMPUTE GPU STATUS\n{separator}\n"
        
        if self.is_available:
            metrics = self._run_nvidia_smi("index,memory.total,memory.used,memory.free,utilization.gpu,temperature.gpu")
            log_entry += f"Final GPU State:\n{metrics}\n"
        
        if summary:
            log_entry += f"\nResult: {summary}\n"
        
        log_entry += f"{separator}\n\n"
        self._append_log(log_entry)
        print(log_entry)
    
    def get_summary(self) -> Dict:
        """Get GPU summary"""
        if not self.is_available:
            return {"status": "unavailable", "mode": "cpu"}
        
        try:
            gpu_info = self._run_nvidia_smi("name,driver_version,memory.total")
            parts = gpu_info.split(', ')
            return {
                "status": "available",
                "mode": "gpu",
                "gpu": parts[0] if len(parts) > 0 else "Unknown",
                "driver": parts[1] if len(parts) > 1 else "Unknown",
                "memory": parts[2] if len(parts) > 2 else "Unknown"
            }
        except:
            return {"status": "error"}
    
    def _append_log(self, message: str):
        """Save log"""
        self.logs.append(message)
        try:
            with open(self.log_file, 'a') as f:
                f.write(message)
        except:
            pass
    
    def get_log_content(self) -> str:
        """Get full log"""
        try:
            with open(self.log_file, 'r') as f:
                return f.read()
        except:
            return "\n".join(self.logs)

# Initialize
gpu_monitor = GPUMonitor("gpu-usage.log")
print("✓ GPU Monitor initialized")

## Load K2Think Client

In [ ]:
import requests
from typing import Optional, List, Dict

class K2ThinkClient:
    """K2Think API Client"""
    
    def __init__(self, 
                 email: Optional[str] = None, 
                 password: Optional[str] = None,
                 api_base: str = "https://www.k2think.ai"):
        self.email = email or os.getenv("K2THINK_EMAIL")
        self.password = password or os.getenv("K2THINK_PASSWORD")
        self.api_base = api_base
        self.token = None
        self.session = requests.Session()
        
        if not self.email or not self.password:
            raise ValueError("K2Think credentials required")
    
    def authenticate(self) -> bool:
        """Authenticate with K2Think"""
        try:
            response = self.session.post(
                f"{self.api_base}/api/auth/login",
                json={"email": self.email, "password": self.password},
                timeout=10
            )
            
            if response.status_code == 200:
                self.token = response.json().get("access_token")
                self.session.headers.update({
                    "Authorization": f"Bearer {self.token}",
                    "Content-Type": "application/json"
                })
                return True
            return False
        except Exception as e:
            print(f"Auth error: {e}")
            return False
    
    def chat_completion(self,
                       model: str = "MBZUAI-IFM/K2-Think",
                       messages: Optional[List] = None,
                       max_tokens: int = 500,
                       temperature: float = 0.7) -> Dict:
        """Chat completion"""
        
        if not self.token and not self.authenticate():
            raise RuntimeError("Authentication failed")
        
        payload = {
            "model": model,
            "messages": messages or [],
            "max_tokens": max_tokens,
            "temperature": temperature,
            "stream": False
        }
        
        try:
            response = self.session.post(
                f"{self.api_base}/api/chat/completions",
                json=payload,
                timeout=30
            )
            return response.json() if response.status_code == 200 else {"error": response.text}
        except Exception as e:
            return {"error": str(e)}

print("✓ K2Think Client loaded")

## 🔑 Enter K2Think Credentials

In [ ]:
from getpass import getpass

# Get credentials
email = os.getenv("K2THINK_EMAIL")
password = os.getenv("K2THINK_PASSWORD")

if not email:
    email = input("Enter K2Think email: ")
if not password:
    password = getpass("Enter K2Think password (will not be shown): ")

print(f"✓ Using K2Think account: {email}")

## 🚀 Run Demo: AI Inference with GPU Monitoring

In [ ]:
print("\n╔════════════════════════════════════════════════════════════╗")
print("║   Custom AI Agent Wrapper - Decentralized Compute          ║")
print("║   K2Think + AIDP GPU Network Demo                          ║")
╚════════════════════════════════════════════════════════════╝")
print()

# Check GPU
gpu_info = gpu_monitor.get_summary()
print(f"GPU Status: {gpu_info['status']}")
if gpu_info['status'] == 'available':
    print(f"  GPU: {gpu_info.get('gpu')}")
    print(f"  Memory: {gpu_info.get('memory')}")
print()

In [ ]:
# Initialize client
try:
    print("🤖 Initializing K2Think client...")
    client = K2ThinkClient(email=email, password=password)
    
    if not client.authenticate():
        print("❌ Authentication failed. Check your credentials.")
    else:
        print("✓ Authenticated\n")
        
        # Pre-compute status
        gpu_monitor.log_pre_compute_status("K2Think AI Inference")
        
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Task 1: Text generation
print("\n⚡ Task 1: Text Generation")
print("-" * 40)
gpu_monitor.log_compute_status("Text Generation")

response = client.chat_completion(
    messages=[{
        "role": "user",
        "content": "What are benefits of GPU-accelerated compute for AI workloads? (2-3 sentences)"
    }],
    max_tokens=150
)

if "error" not in response:
    text = response["choices"][0]["message"]["content"]
    tokens = response.get("usage", {}).get("total_tokens", 0)
    print(f"\nAI Response: {text}")
    print(f"\nTokens used: {tokens}")
else:
    print(f"Error: {response['error']}")

gpu_monitor.log_compute_status("Text Generation Complete")

In [ ]:
# Task 2: Code generation
print("\n⚡ Task 2: Code Generation")
print("-" * 40)
gpu_monitor.log_compute_status("Code Generation")

response = client.chat_completion(
    messages=[{
        "role": "user",
        "content": "Write a simple Python snippet to check GPU availability using subprocess"
    }],
    max_tokens=200
)

if "error" not in response:
    code = response["choices"][0]["message"]["content"]
    tokens = response.get("usage", {}).get("total_tokens", 0)
    print(f"\nAI Response:\n{code}")
    print(f"\nTokens used: {tokens}")
else:
    print(f"Error: {response['error']}")

gpu_monitor.log_compute_status("Code Generation Complete")

In [ ]:
# Task 3: Analysis
print("\n⚡ Task 3: Technical Analysis")
print("-" * 40)
gpu_monitor.log_compute_status("Technical Analysis")

response = client.chat_completion(
    messages=[{
        "role": "user",
        "content": "Explain decentralized GPU compute networks and their importance for AI infrastructure"
    }],
    max_tokens=250
)

if "error" not in response:
    analysis = response["choices"][0]["message"]["content"]
    tokens = response.get("usage", {}).get("total_tokens", 0)
    print(f"\nAI Response:\n{analysis}")
    print(f"\nTokens used: {tokens}")
else:
    print(f"Error: {response['error']}")

gpu_monitor.log_post_compute_status("All GPU compute tasks completed successfully")

## 📋 View GPU Activity Log

In [ ]:
print("\n📋 GPU Activity Log:")
print("=" * 60)
log_content = gpu_monitor.get_log_content()
lines = log_content.split('\n')
for line in lines[-100:]:
    if line.strip():
        print(line)
print("=" * 60)
print(f"\n✅ Total log entries: {len(gpu_monitor.logs)}")

## 🎬 Ready for Submission!

You've successfully demonstrated:
- ✅ GPU detection and monitoring
- ✅ K2Think AI inference with GPU logging
- ✅ Real-time compute metrics

### Next Steps for AIDP Bounty:

1. **Record Demo Video**
   - Scroll up to see GPU logs
   - Take screenshot or record screen
   - Show nvidia-smi output and AI responses
   - Duration: 1-2 minutes

2. **Submit to Superteam Earn**
   - GitHub: https://github.com/HEDELKA/k2think-aidp
   - This notebook link
   - Demo video URL
   - GPU usage explanation

---

**Custom AI Agent Wrapper optimized for Decentralized Compute** 🚀